# Protect-RscWorkload
In some environments where High Availability is required, DBAs will use a feature called Availability Groups. If this is used in an environment, then the Availability Group is the parent of the database. by getting this value first, it will allow you to narrow down the places the database can exist and allowing you to work with the database you need to. 

# Workflow
To get information about the SQL Server Instance, you need to do the following
1. Use **Connect-RSC** to Connect to RSC
1. Use **Get-RscCluster** to get the cluster id that the SQL Server is registered. This is akin to the datacenter that the SQL Server is located in. While you can run the next command without the cluster ID, you may get more records back than expected. This is because your SQL Server Host could exist on 2 different cluster. Once as a replicated source and once as the replicated target. 
1. Use **Get-RscHost** to get the host id that the SQL Server is installed on. 
1. Use **Get-RscMssqlAvailabilityGroup** to get the id of the Availability Group. 
1. Use **Protect-RscLinkedWorkload** to asssign an SLA to the AG
1. Use **Disconnect-RSC** to disconnect from RSC

# Connect to RSC

In [1]:
Import-Module RubrikSecurityCloud
$ConnectRsc = @{
    Server = op read "op://Rubrik/RoadRunnerServiceAccount/hostname"
    ClientId = op read "op://Rubrik/RoadRunnerServiceAccount/username"
    ClientSecret = ConvertTo-SecureString (op read "op://Rubrik/RoadRunnerServiceAccount/credential") -AsPlainText -Force
}
Connect-RSC @ConnectRsc

ServiceAccountDetails


ClientId,client|1461d755-e4d2-4a71-8a5d-204f7d6bc73e
AuthenticationState,AUTHORIZED
SdkSchemaVersion,v20250616-31
ServerVersion,v20250707-25


# Protect AG Non-Linked

## Get the Cluster ID

In [ ]:
$ClusterName = "vault-r-london"
$RscCluster = Get-RscCluster -Name $ClusterName

## Get the Availability Group

In [ ]:
$AvailabilityGroupName = "RR-SQL22-AG2"
$RscMssqlAvailabilityGroup = Get-RscMssqlAvailabilityGroup -AvailabilityGroupName $AvailabilityGroupName -RscCluster $RscCluster -Replica:$false -Relic:$false

## Get the SLA Domain

In [ ]:
$RscSlaDomain = Get-RscSla -Name "MSSQL-1d-30d-NoArchive-Repl-Houghton-London"

## Protect the Availability Group

In [ ]:
Protect-RscWorkload -InputObject $RscMssqlAvailabilityGroup -Sla $RscSlaDomain -AssignmentType PROTECT_WITH_SLA_ID

# Protect AG Linked

## Get the First Cluster

In [2]:
$ClusterName = "vault-r-london"
$RscCluster1 = Get-RscCluster -Name $ClusterName

## Get the Second Cluster

In [3]:
$ClusterName = "vault-f-houghton"
$RscCluster2 = Get-RscCluster -Name $ClusterName

## Get the Availability Groups

In [4]:
$AvailabilityGroupName = "RR-SQL22-AG6"
$RscMssqlAvailabilityGroup1 = Get-RscMssqlAvailabilityGroup -AvailabilityGroupName $AvailabilityGroupName -RscCluster $RscCluster1 -Replica:$false -Relic:$false
$RscMssqlAvailabilityGroup2 = Get-RscMssqlAvailabilityGroup -AvailabilityGroupName $AvailabilityGroupName -RscCluster $RscCluster2 -Replica:$false -Relic:$false

In [ ]:
$RscMssqlAvailabilityGroup1

In [ ]:
$RscMssqlAvailabilityGroup2

## Get the SLA Domain

In [5]:
$RscSlaDomain = Get-RscSla -Name "MSSQL-1d-30d-NoArchive-Repl-Houghton-London"

## Protect the Availability Group with Linkage

In [6]:
Protect-RscLinkedWorkload -InputObject $RscMssqlAvailabilityGroup1 -LinkedObject $RscMssqlAvailabilityGroup2 -LinkingOperation LINK -AssignmentType PROTECT_WITH_SLA_ID -Sla $RscSlaDomain

JobId,32
TaskchainId,0197faa8-eba7-7e5f-a219-d26cb458fd0b


---

# Disconnect from RSC
It is important to have each script you create end by disconnecting from RSC. If you do not, then you will run into issues with token authentication exhaustion. RSC allows a Service Account to have 10 concurrent open connections to RSC. Additional requests will be denied until existing connections are closed. 

You can easily disconnect your connection by running the below statement. 

In [ ]:
Disconnect-Rsc